In [5]:
import json
from pathlib import Path
from typing import List, Tuple, Union
from ures.files import filter_files
from perf_estimator.estimator import Estimator, TrainerEstimator
from perf_estimator.dataset import image_dataset
from experiments.snapshot import SnapshotAnalyser
from perf_estimator.profiler import ProfilerDataProcessing
from ures.string import format_memory

In [6]:
train_on_gpu_data_dir = "/home/glaswigian/.cache/XMemEstimator/tprofiler-VGG16-batch-50-Nvidia"
train_on_cpu_data_dir = "/home/glaswigian/.cache/XMemEstimator/9ad90ed2b5534006ab3c1384c85bb8a6"

In [7]:
class xMemComparer:
    def __init__(self, gpu_data_dir: str, cpu_data_dir: str):
        self._cpu_profiler = filter_files(".pt.trace.json", cpu_data_dir, fuzz=True)[-1]
        self._gpu_snapshot = filter_files(".pickle", gpu_data_dir, fuzz=True)[-1]
        self._gpu_nvml = filter_files("host_metrics-", gpu_data_dir, fuzz=True)[-1]

    def estimated_memory(self, profiler_file: str, batch_size: int, gpu_capacity: Union[int, float]) -> float:
        estimator = TrainerEstimator(
            dataloader=image_dataset(batch=int(batch_size)),
            profiler_file=profiler_file,
            max_gpu_memory_in_gb=gpu_capacity
        )
        my_result, _ = estimator.estimate()
        return max(my_result._trace.max_segment_changes)

    def snapshot_memory(self, snapshot_file: str):
        _snapshot = SnapshotAnalyser(snapshot_file)
        return max(_snapshot.gpu_and_segment_in_same_time_length()['seg'])

    def nvml_memory(self, nvml_file: str):
        with open(nvml_file, "r") as f:
            _data = json.load(f)
        _gpu_memory_usage = {}
        start_memory = {}
        for index, gpu_metric in enumerate(_data["records"]):
            gpu_data = gpu_metric["HostGPUs"]
            for device_id, data in gpu_data.items():
                if index == 0:
                    start_memory[device_id] = data["memory"]["used"]

                if device_id not in _gpu_memory_usage:
                    _gpu_memory_usage[device_id] = []
                _gpu_memory_usage[device_id].append(
                    data["memory"]["used"] - start_memory[device_id]
                )
        return max(_gpu_memory_usage["0"])

    def summary(self, batch_size: int, gpu_capacity: Union[int, float]) -> dict:
        est_max_seg = self.estimated_memory(
            profiler_file=self._cpu_profiler,
            batch_size=batch_size,
            gpu_capacity=gpu_capacity
        )
        snapshot_seg = self.snapshot_memory(self._gpu_snapshot)
        nvml_seg = self.nvml_memory(nvml_file=self._gpu_nvml)
        return {
            "xMem Est": format_memory(est_max_seg),
            "Snapshot Ground": format_memory(snapshot_seg),
            "NVML Ground": format_memory(nvml_seg),
        }



In [8]:
xmem_c = xMemComparer(
    gpu_data_dir=train_on_gpu_data_dir,
    cpu_data_dir=train_on_cpu_data_dir,
)
xmem_c.summary(batch_size=50, gpu_capacity=8)

ValueError: Time 5042112482278.404, the amount of memory freed is not the same as the current block.Original Size: 819200 != free Size: -147456